![Banner](./img/AI_Special_Program_Banner.jpg)


## Exercise for Working with Text Data
---
* Instructions are given in <span style="color:blue">blue</span> color.
* Feel free to insert more cells wherever you see fit.

For this exercise, you will work with some basic techniques typically applied in text mining to develop a minimalistic, text based search engine. To support this task, you will explore some aspects of Python's `nltk` ([Natural Language Toolkit](https://www.nltk.org)) library.

> **Attention:** The tasks marked as optional do not have to be completed. If you want or are interested, you can sumbmit them.

## 1. Retrieving data

The data used in this exercise consists of a small collection of app descriptions scraped from the [Google Play Store](https://play.google.com/store/apps?hl=en). From this collection, your search engine should retrieve those apps whose descriptions best match a given query.

In [1]:
# Packages and libraries
import pandas as pd
import numpy as np
import nltk
nltk.download('wordnet')
nltk.download('stopwords')

from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\psybe\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\psybe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
corpus = pd.read_json('./data/app-descriptions.json')

* <div style="color:blue">As you should have done many times by now, yet again, it is good practice to familiarize yourself with the provided data. Go ahead and find out how many app descriptions there are in total. Please have a look at the structure of the <code>DataFrame</code> by printing out its first five rows. Lastly, print out the full app description for the first app in the <code>DataFrame</code>.</div>

In [3]:
# Total number of app descriptions in the corpus
print(f"Number of app descriptions: {len(corpus)}\n")

# Structure of the DataFrame (first five rows)
display(corpus.head())

# Full description of the first app
print(f"\nFull description of the first app ('{corpus['name'][0]}'):\n")
print(corpus['description'][0])

Number of app descriptions: 1614



,name,description
0,10000000,﻿10000000 is a Dungeon Crawling Puzzle RPG Mat...
1,1177 Vårdguiden,I 1177 Vårdguidens app får du tillgång till 11...
2,123 Draw🎨 Toddler counting for kids Drawing games,Need counting games for kids & drawing for tod...
3,123 Kids Fun MUSIC - Kids Music Educational Games,Beautiful and simple music application for tod...
4,123 number games for kids - Count & Tracing,A Fun and intuitive numbers game for your baby...



Full description of the first app ('10000000'):

﻿10000000 is a Dungeon Crawling Puzzle RPG Matching Game. Run the dungeon, match tiles and run for as long as you can


“This game is so good it makes me want to cry” - TouchArcade


- Monsters blocking your way? Match swords and staves to take down pixel art Orcs, Dragons and Elementals

- Can't see a match? Dig into your backpack for items. How about casting a magic spell that changes all the tiles on the puzzle board, or eating some food to give yourself an energy boost.

- Battle getting too tough? Use the loot you found to go back to the Blacksmith and Armorer to upgrade your gear to epic status


“Freedom costs 10 million. I’m half-hoping I never make it” – Kotaku


- It's an RPG too! - Each dungeon and monster fight gives you experience to spend on new skills at at the trainer

- Upgrade your castle - Matching stone and wood lets you bring it back home to upgrade your castle


“It’s damned good.” – Pocket Tactics


- Brew potions

## 2. Preparing data

* <div style="color:blue">To start things off, you are asked to implement a function called <code>preprocess</code> that takes a string as an argument and returns a list of prepared words/tokens from that string.</div>

    * **Normalize** data (e.g., remove punctuation and transform to lower case characters)
    * **Tokenize** data (tokens should be at least 2 characters long)
    * **Lemmatize** tokens (you don't need to [POS tag](https://en.wikipedia.org/wiki/Part-of-speech_tagging) each token for this exercise - however - if you chose to use the `WordNetLemmatizer`, provided by `nltk`, make sure to tag each token as verb `v`, because the default POS tag (noun `n`) does not yield the best results for this application)
    * Remove stop words
    * Return the prepared list of tokens

**Hint**: All required steps can be executed with the help of `nltk`. Have a look at the libraries loaded at the beginning for ideas which packages could help you to solve the task. However, you can also choose different methods/libraries, as long as you reach your goal. 

In [4]:
# Reusable nltk objects: alphabetic tokenizer (tokens >= 2 chars), verb lemmatizer, English stop words
tokenizer = RegexpTokenizer(r'\b[^\d\W]{2,}\b')
lemmatizer = WordNetLemmatizer()
sws = set(stopwords.words('english'))

def preprocess(text):
    tokens = tokenizer.tokenize(text.lower())                    # normalize (lower case) + tokenize
    lemmas = [lemmatizer.lemmatize(t, pos='v') for t in tokens]  # lemmatize, POS-tagged as verb
    lemmas = [l for l in lemmas if l not in sws]                 # remove English stop words
    return lemmas

* <div style="color:blue">Test your implementation by running the following cell.</div>

In [5]:
preprocess('Vaccination protects a person from disease.\n')

['vaccination', 'protect', 'person', 'disease']

Your output should look like this:

## 3. Generating features (vectorizing data)

* <div style="color:blue">Next, you need to vectorize your tokens - as most machine learning algorithms cannot handle non-numerical data. For this exercise, you are asked to map each app description to a <b>tf-idf</b> vector.</div>

    * Use the [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) class from `scikit-learn` to generate your features
    * `TfidfVectorizer` accepts a `tokenizer` parameter to which you can provide your costum `preprocess` function
    * Don't be confused by the existence of the `preprocessor` parameter - this one handles string transformation and is not needed here
    
<b>Please be patient, this can take quite some time!</b>

In [6]:
# Map each app description to a tf-idf vector, using our custom preprocess as the tokenizer
vectorizer = TfidfVectorizer(tokenizer=preprocess, token_pattern=None)
features = vectorizer.fit_transform(corpus['description'])

* <div style="color:blue">Have a look at the shape of your resulting feature array. What is the interpreation of that shape (what do the values represent)?.</div>

In [7]:
features.shape

(1614, 23676)

*Your (written) solution goes here:*

The feature array has shape **(1614, 23676)**:

* **1614 rows** &mdash; one per app description (document) in the corpus.
* **23676 columns** &mdash; one per unique token in the vocabulary learned from all descriptions.

So each app description is represented as a 23676-dimensional (sparse) tf-idf vector, where every entry is the tf-idf weight of one vocabulary token in that description.

## 4. Building a search engine

Now that you have your prepared features, it is time to implement the search engine. The general goal is to provide a list of apps whose descriptions are most relevant to a given query.
One way to solve this task is to implement the unsupervised [NearestNeighbor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html) learner provided by `scikit-learn`. This learner allows you to easily find the app descriptions closest (nearest) to the vector representation of the provided query.

* <div style="color:blue">Implement a <code>search_apps</code> function that takes a query string as an input argument and returns a <code>DataFrame</code> containing the three (n=3) apps whose descriptions are most similar to the provided query (in terms of <b>cosine</b> similarity).</div>

In [8]:
# Fit an unsupervised nearest-neighbour model on the tf-idf features (cosine distance)
neighbours = NearestNeighbors(n_neighbors=3, metric='cosine')
neighbours.fit(features)

def search_apps(query):
    query_vec = vectorizer.transform([query])              # vectorize query into the same tf-idf space
    distances, indices = neighbours.kneighbors(query_vec)  # the 3 closest app descriptions
    result = corpus.iloc[indices[0]].copy()
    result['cosine_similarity'] = 1 - distances[0]         # cosine similarity = 1 - cosine distance
    return result

* <div style="color:blue">Test your implementation by running the following cell.</div>

In [9]:
search_apps('taking notes to improve productivity')

,name,description,cosine_similarity
1406,"Todoist: To-Do List, Tasks & Reminders","🏆 2019 Editor's Choice by Google\n🥇 ""Todoist i...",0.169405
611,Google Keep - Notes and Lists,Quickly capture what’s on your mind and get a ...,0.115991
1452,Ultimate Watch 2 watch face,One watch face for every occasion.\n\nUse it f...,0.093800


Your best result should be app number 1406: **Todoist**

## 5. Analysing inverse document frequencies

In the lecture, you have learned about the term "inverse document frequency" (idf). This measure depends on how often a particular word can be found in different documents of a collection of texts (e.g., app descriptions).

* <div style="color:blue">Extract the ten tokens that hold the highest idf values and the ten tokens with the lowest idf values from your app description corpus.</div>

**Hint**: Your `TfidfVectorizer` object, created earlier, holds all the idf values (obj.idf_) in the order of the tokens as well as the token names (obj.get_feature_names_out()).

In [10]:
# Pair every token with its idf value
idf = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())

# Ten tokens with the highest and lowest idf values
highest_idf = idf.sort_values(ascending=False).head(10)
lowest_idf = idf.sort_values().head(10)

print("Ten tokens with the HIGHEST idf (rare - appear in very few descriptions):")
print(highest_idf.to_string())
print("\nTen tokens with the LOWEST idf (common - appear in many descriptions):")
print(lowest_idf.to_string())

Ten tokens with the HIGHEST idf (rare - appear in very few descriptions):
ﬂying                       7.693943
__                          7.693943
ﬂip                         7.693943
ﬁrst                        7.693943
________                    7.693943
____________                7.693943
_______________             7.693943
___________________         7.693943
_______________________     7.693943
________________________    7.693943

Ten tokens with the LOWEST idf (common - appear in many descriptions):
game       1.289541
play       1.407014
feature    1.465432
free       1.582476
new        1.669286
use        1.706236
make       1.731650
get        1.753772
com        1.760372
time       1.787220


* <div style="color:blue">What is your interpretation of the results (i.e., why do those tokens have high/low idf values)?</div>

*Your (written) solution goes here:*

Conceptually `idf(t) = log( N / df(t) )`, so a token's idf **grows when it appears in few descriptions** and **shrinks when it appears in many**.

* **High idf** tokens all sit at the same maximum value (&asymp; 7.69), because they occur in only a single description (`df = 1`). About 13,800 tokens are tied at this maximum, so the exact "top ten" is arbitrary and ends up dominated by noise &mdash; OCR/ligature artefacts (`ﬂying`, `ﬁrst`) and rows of underscores. These terms are very specific but too rare to help match most queries.
* **Low idf** tokens (`game`, `play`, `feature`, `free`, `new`, `use`, `make`, `get`, `com`, `time`) show up in a large share of the descriptions. They are generic app-store vocabulary that barely distinguishes one app from another, so tf-idf down-weights them.

In short, idf is high for rare/specific terms and low for frequent/common terms &mdash; which is exactly what lets tf-idf emphasise the distinctive words when comparing a query to the app descriptions.